# Multimodal Masked-Model Test Bed

Test harness for the `MultiMaskSNIPWrapper` / `MultimodalSNIPMask` parametrization
used in `train_script_rev.py` (`src/masked_model.py`).

**What we are verifying**

The design intent is that a single `MultimodalSNIPMask` parametrization registered on
each prunable layer's `weight` handles *both* the forward pass (masked weights) **and**
the backward pass (masked-out weights receive no gradient), in both the unimodal and
multimodal cases. The open question from the design notes:

> *"I thought the parametrization module would take care of both forward and backward
> in unimodal and multimodal case, but it needs to be verified. Maybe `mask_gradients`
> is needed for gradients too."*

This notebook answers that empirically.

**Test plan**

1. **Dependencies / extraction** — import `ResNet3D` and the wrapper exactly as the
   training script does, on a tiny configurable model.
2. **Unimodal**
   1. *Forward leak* — masked-out weights must not influence the output.
   2. *Backward leak* — masked-out weights must receive **zero gradient** and must
      **not move** after an optimizer step.
3. **Multimodal**
   1. *Forward* — each sample is routed through its own modality's mask; modalities
      do not contaminate one another.
   2. *Backward* — weights outside the union of the batch's masks get zero gradient;
      weights shared by several modalities accumulate the **sum** of per-modality
      gradients (verified against a manual per-modality recomputation).
   3. The role of `mask_gradients` is isolated and measured.

Every check is a hard `assert`; a final summary table reports PASS/FAIL.

> The conceptual reason masking "just works" for gradients: the parametrization makes
> the *materialized* weight `w_eff = w_orig * mask`. Autograd therefore sends
> `dL/dw_orig = dL/dw_eff * mask` back to the stored parameter, so any position with
> `mask == 0` gets exactly zero gradient — for free, per sub-forward. The multimodal
> forward runs each modality under its own active mask, so `w_orig.grad` ends up as the
> *sum* over the batch's modalities. We test whether that theory holds in practice, and
> whether `mask_gradients` adds anything on top of it.

## 1. Dependencies & model extraction

We import the real classes from the repo. Set `REPO_ROOT` to the project directory
(the folder containing `resnet.py` and `src/`). The model is built with a **tiny,
configurable** geometry so the whole notebook runs in seconds on CPU while exercising
the exact same code paths (`Conv3d`, `Linear`, `parametrize`, `mask_gradients`) as the
full 64-base / 256³ training config.

In [1]:
import os, sys, itertools, math
from copy import deepcopy

# ---- point this at your project root (contains resnet.py and src/) ----
REPO_ROOT = os.environ.get("REPO_ROOT", os.path.abspath("."))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.nn.utils.parametrize as parametrize

print("torch:", torch.__version__, "| cuda available:", torch.cuda.is_available())

# resnet.py does `from pynvml import ...` at import time (only used by the
# unused enMesh memory-profiling path). If pynvml/nvidia-ml-py isn't installed
# in this env, install a tiny stub so the import succeeds. The test bed never
# touches GPU memory profiling. (`pip install nvidia-ml-py` removes the need.)
try:
    import pynvml  # noqa: F401
except ModuleNotFoundError:
    import types
    _stub = types.ModuleType("pynvml")
    _stub.nvmlDeviceGetHandleByIndex = lambda *a, **k: None
    _stub.nvmlDeviceGetMemoryInfo = lambda *a, **k: None
    sys.modules["pynvml"] = _stub
    print("note: pynvml not found -> installed an in-memory stub for the import.")

# The training script does: `from resnet import ResNet3D`
from resnet import ResNet3D
# and: `from src.masked_model import MultiMaskSNIPWrapper`
from src.masked_model import MultiMaskSNIPWrapper, MultimodalSNIPMask, PRUNE_LAYERS

# Modality string -> integer code. This is the canonical mapping defined in
# src/customMongoDataset.py (map_modality_codes / MultimodalMongoDataset docstring).
# We inline it here so the test bed has no MongoDB / mindfultensors dependency.
MODALITY_CODES = {"smri": 0, "falff": 1, "dwi": 2}
def map_modality_codes(mod):
    return MODALITY_CODES[mod]

print("Prunable layer types:", PRUNE_LAYERS)
print("Modality codes:", {m: map_modality_codes(m) for m in ["smri", "falff", "dwi"]})

DEVICE = torch.device("cpu")          # CPU is fine for the tiny config
torch.manual_seed(0)

/Users/ppopov1/miniconda3/envs/pile/lib/python3.12/site-packages/torch/cuda/__init__.py:65: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


torch: 2.10.0 | cuda available: False
Prunable layer types: (<class 'torch.nn.modules.linear.Linear'>, <class 'torch.nn.modules.conv.Conv3d'>)
Modality codes: {'smri': 0, 'falff': 1, 'dwi': 2}


In [2]:
# ---- Tiny, configurable test geometry --------------------------------------
# The real config is channels=64, volume=256^3. That is far too heavy to iterate
# on. These knobs drive an identical-architecture miniature. Scale them up on a
# GPU box if you want to stress the real thing.
CHANNELS      = 4          # base channel count (real: 64)
VOLUME        = 16         # cube side length  (real: 256)
SPARSITY      = 0.9        # fraction of weights masked OUT (real default: 0.9)
N_PER_MOD     = 2          # samples per modality in a batch

# Modalities under test (codes as produced by torch.unique(modality) in training)
MODS = [map_modality_codes("smri"), map_modality_codes("falff")]   # [0, 1]
print("Testing modalities (int codes):", MODS)

def build_base_model():
    torch.manual_seed(0)
    m = ResNet3D(in_channels=1, n_classes=1, channels=CHANNELS).to(DEVICE)
    return m

def random_batch(mods, n_per_mod=N_PER_MOD, seed=None):
    '''Build a synthetic multimodal batch matching the training data layout:
    inputs  : (B, 1, V, V, V) float
    modality: (B,) long  -- integer modality codes
    labels  : (B, 1) long -- binary
    '''
    if seed is not None:
        torch.manual_seed(seed)
    xs, ms, ys = [], [], []
    for mod in mods:
        xs.append(torch.randn(n_per_mod, 1, VOLUME, VOLUME, VOLUME, device=DEVICE))
        ms.append(torch.full((n_per_mod,), int(mod), dtype=torch.long, device=DEVICE))
        ys.append(torch.randint(0, 2, (n_per_mod, 1), device=DEVICE))
    return torch.cat(xs), torch.cat(ms), torch.cat(ys)

_m = build_base_model()
n_params = sum(p.numel() for p in _m.parameters())
n_prunable = sum(1 for _, mod in _m.named_modules() if isinstance(mod, PRUNE_LAYERS))
print(f"Base model: {n_params:,} params | {n_prunable} prunable (Conv3d/Linear) layers")
del _m

Testing modalities (int codes): [0, 1]
Base model: 131,413 params | 21 prunable (Conv3d/Linear) layers


In [3]:
# ---- Build a masked model with real SNIP-initialized masks -----------------
# This follows get_model() in train_script_rev.py exactly: wrap, then
# register_multimodal_masks(modalities, input_data, labels) using a SNIP batch.
def make_masked_model(mods=MODS, sparsity=SPARSITY, seed=0):
    torch.manual_seed(seed)
    base = build_base_model()
    wrapped = MultiMaskSNIPWrapper(base, sparsity=sparsity).to(DEVICE)
    # SNIP "calibration" batch -- one sub-batch per modality, exactly like get_snip_data
    snip_x, snip_mod, snip_y = random_batch(mods, n_per_mod=N_PER_MOD, seed=seed + 1)
    wrapped.register_multimodal_masks(snip_mod, snip_x, snip_y)
    return wrapped

model = make_masked_model()
print("masks_registered:", model.masks_registered)

# Inspect which layers got parametrized and the realized sparsity per modality
print("\nLayer / modality mask sparsity (fraction of weights == 0):")
for name, module in model.model.named_modules():
    if parametrize.is_parametrized(module, "weight"):
        pm = [p for p in module.parametrizations.weight if isinstance(p, MultimodalSNIPMask)][0]
        sps = {k: 1 - getattr(pm, f"mask_{k}").float().mean().item() for k in pm.keys}
        print(f"  {name:<28} {sps}")

Generating SNIP masks for modality: 0
Generating SNIP masks for modality: 1
Registering Parametrizations...
Mask initialization complete. Temporary CPU model cleared.
masks_registered: True

Layer / modality mask sparsity (fraction of weights == 0):
  conv1                        {0: 0.0007288455963134766, 1: 0.0}
  layer1.0.conv1               {0: 0.0, 1: 0.0}
  layer1.0.conv2               {0: 0.0, 1: 0.0}
  layer1.1.conv1               {0: 0.0, 1: 0.0}
  layer1.1.conv2               {0: 0.0, 1: 0.0}
  layer2.0.conv1               {0: 0.009259283542633057, 1: 0.009259283542633057}
  layer2.0.conv2               {0: 0.21296298503875732, 1: 0.17129629850387573}
  layer2.0.shortcut.0          {0: 0.0, 1: 0.0}
  layer2.1.conv1               {0: 0.1527777910232544, 1: 0.13425928354263306}
  layer2.1.conv2               {0: 0.11168980598449707, 1: 0.2222222089767456}
  layer3.0.conv1               {0: 0.7664930522441864, 1: 0.8217592537403107}
  layer3.0.conv2               {0: 0.962962962

### Helpers

Small utilities used by every test: enumerate parametrized layers, fetch a layer's
stored ("original") weight and a given modality's mask, and a tiny assertion logger
that records results for the final summary.

In [4]:
RESULTS = []  # (section, name, passed, detail)

def check(section, name, passed, detail=""):
    RESULTS.append((section, name, bool(passed), detail))
    flag = "PASS" if passed else "FAIL"
    print(f"  [{flag}] {name}" + (f"  --  {detail}" if detail else ""))
    assert passed, f"{section} / {name} FAILED: {detail}"

def parametrized_layers(m):
    '''Yield (name, module, MultimodalSNIPMask) for every masked layer.'''
    for name, module in m.model.named_modules():
        if parametrize.is_parametrized(module, "weight"):
            for pm in module.parametrizations.weight:
                if isinstance(pm, MultimodalSNIPMask):
                    yield name, module, pm

def original_weight(module):
    # The raw, un-masked stored parameter behind the parametrization
    return module.parametrizations.weight.original

def mask_for(pm, mod):
    return getattr(pm, f"mask_{int(mod)}")

# sanity: there is at least one masked layer and masks are genuinely sparse
_layers = list(parametrized_layers(model))
assert _layers, "No parametrized layers found -- mask registration failed!"
print(f"{len(_layers)} parametrized layers available for testing.")

21 parametrized layers available for testing.


## 2. Unimodal case

A batch containing a **single** modality. We use modality `MODS[0]`.

In [5]:
UNI_MOD = MODS[0]
uni_x, uni_mod, uni_y = random_batch([UNI_MOD], n_per_mod=N_PER_MOD, seed=42)
print("unimodal batch:", uni_x.shape, "| modality codes:", uni_mod.tolist(), "| labels:", uni_y.flatten().tolist())

unimodal batch: torch.Size([2, 1, 16, 16, 16]) | modality codes: [0, 0] | labels: [0, 1]


### 2.1 Forward leak test

**Claim:** with modality `m` active, the effective weight of every masked layer is
`w_orig * mask_m`, so positions where `mask_m == 0` contribute nothing to the output.

**Two independent checks:**

- *Materialization:* read back `module.weight` while `m` is active and assert it equals
  `w_orig * mask_m` exactly (and that masked positions are exactly 0).
- *Perturbation (the strong test):* deliberately corrupt the masked-out entries of the
  *stored* weights with large garbage values. If the masked weights truly never reach
  the forward pass, the output must be **bitwise unchanged**. If the output moves, a
  masked weight leaked.

In [6]:
# --- Check A: materialized weight == original * mask, masked entries are exactly 0
model.eval()
model._set_active_modality(UNI_MOD)
max_err = 0.0
for name, module, pm in parametrized_layers(model):
    w_eff  = module.weight                      # triggers parametrization forward
    w_orig = original_weight(module)
    mask   = mask_for(pm, UNI_MOD)
    err = (w_eff - w_orig * mask).abs().max().item()
    max_err = max(max_err, err)
    leaked = (w_eff[mask == 0]).abs().max().item() if (mask == 0).any() else 0.0
    assert leaked == 0.0, f"{name}: nonzero weight at masked position"
model._set_active_modality(None)
check("2.1 forward", "materialized weight == w_orig * mask_m", max_err == 0.0,
      f"max|w_eff - w_orig*mask| = {max_err:.2e}")

  [PASS] materialized weight == w_orig * mask_m  --  max|w_eff - w_orig*mask| = 0.00e+00


In [7]:
# --- Check B: perturb masked-out stored weights -> forward output must NOT change
model.eval()
with torch.no_grad():
    y_ref = model.forward(uni_x, uni_mod).clone()

# Corrupt every masked-out (mask_m == 0) entry of the stored weights with big garbage.
with torch.no_grad():
    for name, module, pm in parametrized_layers(model):
        w_orig = original_weight(module)
        mask = mask_for(pm, UNI_MOD)
        w_orig.add_((1.0 - mask) * 1.1)   # blow up only the masked-out positions

with torch.no_grad():
    y_pert = model.forward(uni_x, uni_mod)

drift = (y_ref - y_pert).abs().max().item()
check("2.1 forward", "masked weights do NOT leak into forward",
      drift == 0.0, f"max|y_ref - y_perturbed| = {drift:.2e} (corrupted masked weights by +1.1)")

# rebuild a clean model for subsequent tests
model = make_masked_model()

  [PASS] masked weights do NOT leak into forward  --  max|y_ref - y_perturbed| = 0.00e+00 (corrupted masked weights by +1.1)
Generating SNIP masks for modality: 0
Generating SNIP masks for modality: 1
Registering Parametrizations...
Mask initialization complete. Temporary CPU model cleared.


### 2.2 Backward leak test

**Claim:** after `loss.backward()`, every masked-out weight has **zero gradient**, and
after an optimizer step those weights have **not moved** — so a modality cannot corrupt
weights it never used.

We test the parametrization *on its own first* (no `mask_gradients` call) to see whether
it is sufficient, then we step the optimizer and confirm masked weights are frozen.

In [8]:
# --- Backward with NO mask_gradients call: are masked-out grads already zero?
model = make_masked_model()
model.train()
opt = torch.optim.Adam(model.parameters(), lr=1e-2)
opt.zero_grad()

y = model.forward(uni_x, uni_mod)
loss = F.binary_cross_entropy_with_logits(y, uni_y.float())
loss.backward()
# NOTE: deliberately NOT calling model.mask_gradients(...) here.

worst = 0.0
total_masked = 0
for name, module, pm in parametrized_layers(model):
    g = original_weight(module).grad
    mask = mask_for(pm, UNI_MOD)
    if g is None:
        continue
    masked_grad = g[mask == 0]
    if masked_grad.numel():
        worst = max(worst, masked_grad.abs().max().item())
        total_masked += masked_grad.numel()
check("2.2 backward", "parametrization alone zeros masked-out grads (no mask_gradients)",
      worst == 0.0, f"max|grad| over {total_masked:,} masked positions = {worst:.2e}")

Generating SNIP masks for modality: 0
Generating SNIP masks for modality: 1
Registering Parametrizations...
Mask initialization complete. Temporary CPU model cleared.
  [PASS] parametrization alone zeros masked-out grads (no mask_gradients)  --  max|grad| over 117,732 masked positions = 0.00e+00


In [9]:
# --- Optimizer step: masked-out weights must not move ------------------------
# Snapshot stored weights, step Adam, compare masked-out positions.
before = {name: original_weight(module).detach().clone()
          for name, module, pm in parametrized_layers(model)}
opt.step()

max_move = 0.0
for name, module, pm in parametrized_layers(model):
    mask = mask_for(pm, UNI_MOD)
    w_now = original_weight(module)
    moved = (w_now - before[name])[mask == 0]
    if moved.numel():
        max_move = max(max_move, moved.abs().max().item())
check("2.2 backward", "masked-out weights frozen after optimizer.step()",
      max_move == 0.0, f"max move at masked positions = {max_move:.2e}")

  [PASS] masked-out weights frozen after optimizer.step()  --  max move at masked positions = 0.00e+00


In [10]:
# --- Does mask_gradients change anything in the unimodal case? ---------------
# Recompute grads, capture them, then call mask_gradients(unique modalities) and
# verify the gradient tensors are identical -> i.e. mask_gradients is redundant
# with the parametrization here (it only re-zeros already-zero entries).
model.zero_grad(set_to_none=True)
y = model.forward(uni_x, uni_mod)
loss = F.binary_cross_entropy_with_logits(y, uni_y.float())
loss.backward()

before_mg = {name: original_weight(module).grad.detach().clone()
             for name, module, pm in parametrized_layers(model)
             if original_weight(module).grad is not None}
model.mask_gradients(torch.unique(uni_mod))
max_delta = max((original_weight(module).grad - before_mg[name]).abs().max().item()
                for name, module, pm in parametrized_layers(model)
                if name in before_mg)
check("2.2 backward", "mask_gradients is a no-op on grads in unimodal case",
      max_delta == 0.0, f"max grad change from mask_gradients = {max_delta:.2e}")

  [PASS] mask_gradients is a no-op on grads in unimodal case  --  max grad change from mask_gradients = 0.00e+00


## 3. Multimodal case

A batch containing **several** modalities at once (`MODS = [0, 1]`). The wrapper's
`forward` splits the batch by modality, runs each sub-batch under its own active mask,
and writes results back into the right rows.

In [11]:
mm_x, mm_mod, mm_y = random_batch(MODS, n_per_mod=N_PER_MOD, seed=7)
print("multimodal batch:", mm_x.shape, "| modality codes:", mm_mod.tolist())

multimodal batch: torch.Size([4, 1, 16, 16, 16]) | modality codes: [0, 0, 1, 1]


### 3.1 Forward routing test

**Claim:** sample `i` (modality `m_i`) is processed with `mask_{m_i}` and nothing else.
We verify by reproducing the wrapper's output manually: for each modality, set it active
and run the bare `model` on just that modality's rows, then scatter into place. The
wrapper output must match row-for-row.

In [12]:
model = make_masked_model()
model.eval()
with torch.no_grad():
    y_wrapper = model.forward(mm_x, mm_mod)

    # Manual reconstruction of the per-modality routing
    y_manual = torch.zeros_like(y_wrapper)
    for mod in torch.unique(mm_mod).tolist():
        idx = (mm_mod == mod)
        model._set_active_modality(mod)
        y_manual[idx] = model.model(mm_x[idx])
    model._set_active_modality(None)

err = (y_wrapper - y_manual).abs().max().item()
check("3.1 forward", "wrapper routes each sample through its modality's mask",
      err == 0.0, f"max|y_wrapper - y_manual| = {err:.2e}")

Generating SNIP masks for modality: 0
Generating SNIP masks for modality: 1
Registering Parametrizations...
Mask initialization complete. Temporary CPU model cleared.
  [PASS] wrapper routes each sample through its modality's mask  --  max|y_wrapper - y_manual| = 0.00e+00


In [13]:
# --- Cross-modality forward isolation ---------------------------------------
# Corrupt weights that are masked OUT for modality A but used by modality B.
# Modality A's outputs must not change; modality B's may.
model = make_masked_model()
model.eval()
A, B = MODS[0], MODS[1]
with torch.no_grad():
    y0 = model.forward(mm_x, mm_mod).clone()
    # perturb positions that are 0 in mask_A (regardless of mask_B)
    for name, module, pm in parametrized_layers(model):
        w = original_weight(module)
        maskA = mask_for(pm, A)
        w.add_((1.0 - maskA) * 1e6)
    y1 = model.forward(mm_x, mm_mod)

idxA = (mm_mod == A)
driftA = (y0[idxA] - y1[idxA]).abs().max().item()
check("3.1 forward", "modality A outputs unaffected by A's masked-out weights",
      driftA == 0.0, f"max drift on modality-A rows = {driftA:.2e}")
model = make_masked_model()

Generating SNIP masks for modality: 0
Generating SNIP masks for modality: 1
Registering Parametrizations...
Mask initialization complete. Temporary CPU model cleared.
  [PASS] modality A outputs unaffected by A's masked-out weights  --  max drift on modality-A rows = 0.00e+00
Generating SNIP masks for modality: 0
Generating SNIP masks for modality: 1
Registering Parametrizations...
Mask initialization complete. Temporary CPU model cleared.


### 3.2 Backward test — union masking & gradient addition

Two claims:

1. **Union masking:** weights that are masked out for **every** modality in the batch
   (outside the union of masks) receive zero gradient.
2. **Gradient addition:** a weight shared by several modalities accumulates the **sum**
   of the per-modality gradient contributions. We verify this against a manual
   recomputation that backprops each modality separately and adds the gradients.

In [14]:
# --- Claim 2 first: multimodal grad == sum of per-modality grads -------------
# Reference: one combined backward through the wrapper.
model = make_masked_model()
model.train()
model.zero_grad(set_to_none=True)
y = model.forward(mm_x, mm_mod)
# Use SUM reduction so splitting the batch by modality and summing is exact
loss = F.binary_cross_entropy_with_logits(y, mm_y.float(), reduction="sum")
loss.backward()
combined_grad = {name: original_weight(module).grad.detach().clone()
                 for name, module, pm in parametrized_layers(model)}

# Manual: backward each modality's sub-batch separately, accumulate grads.
manual_grad = {name: torch.zeros_like(g) for name, g in combined_grad.items()}
for mod in torch.unique(mm_mod).tolist():
    idx = (mm_mod == mod)
    model.zero_grad(set_to_none=True)
    model._set_active_modality(mod)
    y_sub = model.model(mm_x[idx])
    l_sub = F.binary_cross_entropy_with_logits(y_sub, mm_y[idx].float(), reduction="sum")
    l_sub.backward()
    model._set_active_modality(None)
    for name, module, pm in parametrized_layers(model):
        g = original_weight(module).grad
        if g is not None:
            manual_grad[name] += g.detach()

max_diff = max((combined_grad[n] - manual_grad[n]).abs().max().item() for n in combined_grad)
check("3.2 backward", "multimodal grad == sum of per-modality grads (gradients added correctly)",
      max_diff < 1e-5, f"max|combined - sum_of_parts| = {max_diff:.2e}")

Generating SNIP masks for modality: 0
Generating SNIP masks for modality: 1
Registering Parametrizations...
Mask initialization complete. Temporary CPU model cleared.
  [PASS] multimodal grad == sum of per-modality grads (gradients added correctly)  --  max|combined - sum_of_parts| = 0.00e+00


In [15]:
# --- Claim 1: weights outside the union of batch masks get zero gradient -----
model = make_masked_model()
model.train()
model.zero_grad(set_to_none=True)
y = model.forward(mm_x, mm_mod)
loss = F.binary_cross_entropy_with_logits(y, mm_y.float())
loss.backward()
# (no mask_gradients yet -- testing the parametrization alone)

worst_union = 0.0
n_outside = 0
for name, module, pm in parametrized_layers(model):
    g = original_weight(module).grad
    if g is None:
        continue
    # union mask over all modalities present in the batch
    union = torch.zeros_like(g, dtype=torch.bool)
    for mod in torch.unique(mm_mod).tolist():
        union |= mask_for(pm, mod).bool()
    outside = g[~union]
    if outside.numel():
        worst_union = max(worst_union, outside.abs().max().item())
        n_outside += outside.numel()
check("3.2 backward", "weights outside batch's union-of-masks get zero grad (parametrization alone)",
      worst_union == 0.0, f"max|grad| over {n_outside:,} out-of-union positions = {worst_union:.2e}")

Generating SNIP masks for modality: 0
Generating SNIP masks for modality: 1
Registering Parametrizations...
Mask initialization complete. Temporary CPU model cleared.
  [PASS] weights outside batch's union-of-masks get zero grad (parametrization alone)  --  max|grad| over 116,727 out-of-union positions = 0.00e+00


In [16]:
# --- Isolate the effect of mask_gradients in the multimodal case -------------
# Compare grads BEFORE and AFTER mask_gradients(unique modalities). If the
# parametrization already zeroed everything outside the union, this is a no-op.
grad_before = {name: original_weight(module).grad.detach().clone()
               for name, module, pm in parametrized_layers(model)
               if original_weight(module).grad is not None}
model.mask_gradients(torch.unique(mm_mod))
max_change = max((original_weight(module).grad - grad_before[name]).abs().max().item()
                 for name, module, pm in parametrized_layers(model)
                 if name in grad_before)
check("3.2 backward", "mask_gradients(batch mods) is a no-op given the parametrization",
      max_change == 0.0, f"max grad change = {max_change:.2e}  -> parametrization already masks grads")

  [PASS] mask_gradients(batch mods) is a no-op given the parametrization  --  max grad change = 0.00e+00  -> parametrization already masks grads


## 4. Summary

In [17]:
import textwrap
print("="*78)
print(f"{'SECTION':<16}{'CHECK':<52}{'RESULT'}")
print("-"*78)
for section, name, passed, detail in RESULTS:
    print(f"{section:<16}{name[:50]:<52}{'PASS' if passed else 'FAIL'}")
print("="*78)
n_pass = sum(1 for _, _, p, _ in RESULTS if p)
print(f"{n_pass}/{len(RESULTS)} checks passed.")

print(textwrap.dedent('''
Findings
--------
* FORWARD (unimodal & multimodal): the MultimodalSNIPMask parametrization fully
  controls the forward pass. Masked-out weights cannot influence the output, and the
  multimodal wrapper routes every sample through its own modality's mask correctly.

* BACKWARD (unimodal & multimodal): the parametrization ALONE already zeros the
  gradient at every masked-out position (because dL/dw_orig = dL/dw_eff * mask).
  In the multimodal case gradients from each modality's sub-forward are summed onto
  the shared stored weight, and positions outside the batch's union-of-masks get
  exactly zero gradient. => mask_gradients() is REDUNDANT for gradient masking given
  this parametrization; in our tests it changes no gradient values.

* CAVEAT (momentum): neither the parametrization nor mask_gradients resets optimizer
  state. With Adam, a weight can still drift on steps where its gradient is zero,
  because of accumulated momentum -- a genuine cross-step, cross-modality leak vector.
  mask_gradients does not fix this. For strict isolation, reset/mask the optimizer
  state per modality (or use per-modality optimizers).

* NOTE: BatchNorm affine params and running statistics are NOT masked (only Conv3d /
  Linear weights are). They are shared across modalities by design -- keep in mind if
  full per-modality isolation is the goal.
'''))

SECTION         CHECK                                               RESULT
------------------------------------------------------------------------------
2.1 forward     materialized weight == w_orig * mask_m              PASS
2.1 forward     masked weights do NOT leak into forward             PASS
2.2 backward    parametrization alone zeros masked-out grads (no m  PASS
2.2 backward    masked-out weights frozen after optimizer.step()    PASS
2.2 backward    mask_gradients is a no-op on grads in unimodal cas  PASS
3.1 forward     wrapper routes each sample through its modality's   PASS
3.1 forward     modality A outputs unaffected by A's masked-out we  PASS
3.2 backward    multimodal grad == sum of per-modality grads (grad  PASS
3.2 backward    weights outside batch's union-of-masks get zero gr  PASS
3.2 backward    mask_gradients(batch mods) is a no-op given the pa  PASS
10/10 checks passed.

Findings
--------
* FORWARD (unimodal & multimodal): the MultimodalSNIPMask parametrization fu